# 04 — End-to-End Pipeline (Preprocess → Train → Infer → Postprocess)



## Objective

Run preprocess → train → infer → postprocess end-to-end. Compare
latency/throughput against a PyTorch baseline. Includes a
`torch.cuda.memory_summary()` alternative via NexusRT metrics.


In [ ]:
# --- Kaggle / fresh-runtime setup ---
import os, shutil, sys, subprocess
from pathlib import Path

SETUP_LOG = Path('/kaggle/working/nexusrt_setup.log')


def _is_repo_root(root: Path) -> bool:
    return (root / 'packaging' / 'pyproject.toml').exists() and (root / 'python' / 'nexusrt').exists()


def _find_repo_root() -> Path:
    input_candidates = [Path('/kaggle/input/nexusrt'), Path('/kaggle/input/nexusrt/nexusrt')]
    if Path('/kaggle/input').exists():
        input_candidates.extend(p.parent.parent for p in Path('/kaggle/input').glob('**/packaging/pyproject.toml'))
    for root in input_candidates:
        if _is_repo_root(root):
            return root

    working_candidates = [Path.cwd(), Path('/kaggle/working/nexusrt')]
    if Path('/kaggle/working').exists():
        working_candidates.extend(p.parent.parent for p in Path('/kaggle/working').glob('**/packaging/pyproject.toml'))
    for root in working_candidates:
        if _is_repo_root(root):
            return root
    raise RuntimeError('NexusRT repo not found. Attach the private Kaggle dataset rbrtsl/nexusrt first.')


def _tail(text: str, n: int = 120) -> str:
    lines = text.splitlines()
    return '\n'.join(lines[-n:])


def _run(cmd, *, check=True, env=None):
    cmd = [str(x) for x in cmd]
    header = '+ ' + ' '.join(cmd)
    print(header)
    with SETUP_LOG.open('a') as f:
        f.write('\n' + header + '\n')
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    if proc.stdout:
        with SETUP_LOG.open('a') as f:
            f.write(proc.stdout)
        print(_tail(proc.stdout, 80))
    if check and proc.returncode:
        print(f'Command failed with exit code {proc.returncode}. Full log: {SETUP_LOG}')
        if proc.stdout:
            print('--- last setup log lines ---')
            print(_tail(proc.stdout, 160))
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout)
    return proc


SETUP_LOG.write_text('NexusRT Kaggle setup log\n')
print('NexusRT Kaggle setup: direct CUDA runtime refresh v4')
src_repo = _find_repo_root().resolve()
work_repo = Path('/kaggle/working/nexusrt')
if str(src_repo).startswith('/kaggle/input/'):
    if work_repo.exists():
        shutil.rmtree(work_repo)
    shutil.copytree(src_repo, work_repo, ignore=shutil.ignore_patterns('.venv', 'packaging/build', '__pycache__', '.pytest_cache'))
    repo = work_repo
else:
    repo = src_repo

os.chdir(repo)
print(f'NexusRT repo: {repo}')
_run(['nvidia-smi'], check=False)
_run(['nvcc', '--version'], check=False)

# Keep source imports available even if editable install fails.
python_dir = str(repo / 'python')
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)
os.environ['PYTHONPATH'] = python_dir + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ.setdefault('NEXUSRT_BOOT_TRACE', '1')

# Kaggle validation uses direct source imports plus the locally built C++ core.
# This avoids editable-install/build-isolation failures in private dataset runs.
print('Using source import + direct CUDA CMake build; skipping editable pip install.')

# Build the CUDA shared library directly so nexusrt._abi can load it.
build_dir = repo / 'packaging' / 'build'
configure_cmd = [
    'cmake', '-S', 'packaging', '-B', str(build_dir), '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    '-DNEXUSRT_ENABLE_CUDA=ON',
    '-DNEXUSRT_ENABLE_METAL=OFF',
    '-DNEXUSRT_BUILD_TESTS=OFF',
    '-DCMAKE_CUDA_ARCHITECTURES=60;75;80;90',
]
configure_ok = _run(configure_cmd, check=False).returncode == 0
if not configure_ok:
    raise RuntimeError(f'CUDA CMake configure failed. These notebooks require CUDA; see {SETUP_LOG}')

build_ok = _run(['cmake', '--build', str(build_dir), '--parallel', '2'], check=False).returncode == 0
if not build_ok:
    raise RuntimeError(f'CUDA CMake build failed. These notebooks require CUDA; see {SETUP_LOG}')

lib_candidates = [
    build_dir / 'outputs' / 'lib' / 'libnexusrt.so',
    build_dir / 'outputs' / 'lib' / 'libnexusrt.dylib',
]
for lib in lib_candidates:
    if lib.exists():
        os.environ['NEXUSRT_LIB'] = str(lib)
        print(f'NEXUSRT_LIB={lib}')
        break
else:
    raise RuntimeError(f'Built NexusRT, but libnexusrt was not found under {build_dir}/outputs/lib. See {SETUP_LOG}')
import nexusrt
info = nexusrt.firmware.init('auto')
print(f'Vendor      : {info.vendor}')
print(f'Arch        : {info.arch}')
print(f'Name        : {info.name}')
print(f'HBM (GB)    : {info.hbm_bytes >> 30}')
print(f'SMs         : {info.sm_count}')
print(f'Features    : {info.features}')



In [ ]:

# --- NexusRT end-to-end pipeline ---
import time
corpus_path = "/kaggle/input/dataset/corpus.txt"  # adjust per dataset
try:
    bytes_read = nexusrt.pipeline.run_preprocess(corpus_path, 1 << 20)
    print(f"Preprocess: read {bytes_read} bytes via GDS")
except Exception as e:
    print(f"Preprocess skipped: {e}")

t0 = time.perf_counter()
out = nexusrt.pipeline.run_infer([1,2,3,4,5], max_new_tokens=32)
t1 = time.perf_counter()
text = nexusrt.pipeline.run_postprocess(out)
nexusrt_us = (t1 - t0) * 1e6
print(f"Infer: {len(out)-5} tokens in {nexusrt_us:.1f} us "
      f"({(len(out)-5) / ((t1-t0)):.0f} tok/s)")
print(f"Output: {text!r}")


In [ ]:

# --- PyTorch baseline (only the baseline uses torch; never imported in src/) ---
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2").cuda()
    inp = tok("Hello, NexusRT.", return_tensors="pt").to("cuda")
    t0 = time.perf_counter()
    with torch.no_grad():
        out_pt = model.generate(**inp, max_new_tokens=32)
    t1 = time.perf_counter()
    pt_us = (t1 - t0) * 1e6
    print(f"PyTorch: {32} tokens in {pt_us:.1f} us "
          f"({32 / (t1-t0):.0f} tok/s)")
    print(f"Speedup (PyTorch / NexusRT): {pt_us / nexusrt_us:.2f}x")
except Exception as e:
    print(f"PyTorch baseline skipped: {e}")


In [ ]:

# --- torch.cuda.memory_summary() alternative via NexusRT metrics ---
# NexusRT does not import torch. We expose the equivalent summary through
# the C ABI:
import ctypes
buf = ctypes.create_string_buffer(1 << 16)
try:
    rc = nexusrt._abi.nexusrt_metrics_dump(nexusrt.firmware._default._ctx,
                                           buf, ctypes.sizeof(buf))
    nexusrt._abi.check(rc)
    print(buf.value.decode())
except Exception as e:
    print(f"metrics dump unavailable: {e}")


In [ ]:

nexusrt.firmware.shutdown()
print("✓ End-to-end pipeline test complete.")
